# Exercise 2.1 — Histograms and boxplots (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day2/day2-00-histograms-boxplots.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

## Overview

Get comfortable with key visualization plots to udnerstand data distribution, histograms and boxplots. Histograms show the overall shape and spread of the data distribution and box plots summarise key statistics (median, spread, and potential outliers) and are particularly helpful for comparing distributions across groups.

The same practical again, in Python, without the commentary --- the ideas are the ones in the R tab. `matplotlib` draws the figures and `seaborn` adds the grouped plots; `pandas` does the summary statistics.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

The R side loads `gapminder` as a package. On the Python side the same table is in the data folder you downloaded, so read it from there:

In [ ]:
gapminder = pd.read_csv("data/gapminder.csv")
gapminder.head()

### Summary statistics

In [ ]:
data = pd.DataFrame({"age": [2, 45, 56, 47, 67, 67, 68, 72,
                             75, 80, 85, 86, 89, 125]})
data.head()

In [ ]:
Mean   = data["age"].mean()
Median = data["age"].median()
Sd     = data["age"].std()          # ddof=1, matching R's sd()

print(Mean, Median, Sd)

Quantiles, then the interquartile range as `Q3 - Q1`:

In [ ]:
Quantile = data["age"].quantile([0, .25, .5, .75, 1])
print(Quantile)

IQR = Quantile[0.75] - Quantile[0.25]
print(IQR)

All of it in one go --- `describe()` is the equivalent of `summary()`:

In [ ]:
data["age"].describe()

### Histogram

In [ ]:
fig, ax = plt.subplots()
ax.hist(data["age"], bins=30)
ax.set_xlabel("age")
plt.show()

Change the bin width and watch the shape change:

In [ ]:
fig, ax = plt.subplots()
ax.hist(data["age"], bins=np.arange(0, 151, 30))
ax.set_xlabel("age")
plt.show()

Now add the mean and the median:

In [ ]:
fig, ax = plt.subplots()
ax.hist(data["age"], bins=30)
ax.axvline(Mean, color="red", linestyle="--")
ax.axvline(Median, color="blue", linestyle="--")
ax.set_xlabel("age")
plt.show()

### Box plot

`vert=False` is the equivalent of `coord_flip()` --- take it out and see.

In [ ]:
fig, ax = plt.subplots()
box = ax.boxplot(data["age"], vert=False)
ax.set_xlabel("age")
plt.show()

The mean is not part of a box plot, so add it yourself:

In [ ]:
fig, ax = plt.subplots()
box = ax.boxplot(data["age"], vert=False)
ax.axvline(Mean, color="blue", linestyle="--")
ax.set_xlabel("age")
plt.show()

`stripplot` is the equivalent of `geom_jitter()`: the points, moved sideways at random so they do not sit on top of each other.

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(x=data["age"], ax=ax, width=0.3)
sns.stripplot(x=data["age"], ax=ax, color="black", alpha=0.4, jitter=0.1)
ax.axvline(Mean, color="blue", linestyle="--")
plt.show()

### By group

In [ ]:
rng = np.random.default_rng(123)
grouped = pd.DataFrame({
    "Group": np.repeat(["A", "B", "C"], 200),
    "Value": np.concatenate([rng.normal(5, 1.0, 200),
                             rng.normal(6, 1.2, 200),
                             rng.normal(7, 1.5, 200)]),
})

Side by side, which is what `p1 + p2` does with patchwork:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

sns.boxplot(data=grouped, x="Group", y="Value", hue="Group",
            width=0.2, ax=ax1,
            flierprops=dict(markerfacecolor="red", markersize=4))
sns.stripplot(data=grouped, x="Group", y="Value",
              color="black", alpha=0.4, ax=ax1)
ax1.set_title("Boxplot with jitter")

for g, d in grouped.groupby("Group"):
    ax2.hist(d["Value"], bins=np.arange(0, 13, 0.3), alpha=0.6,
             edgecolor="black", label=g)
ax2.legend(title="Group")
ax2.set_title("Histogram of Values by Group")

plt.show()

One above the other is `p1 / p2`: the same call with `subplots(2, 1)`.

### gapminder

In [ ]:
g = sns.catplot(data=gapminder, x="continent", y="lifeExp", hue="continent",
                col="year", col_wrap=4, kind="box", height=2.2)
g.set_xticklabels(rotation=45, ha="right")
g.set_titles("{col_name}")
plt.show()

Then the same question: for 1952, in Africa, does the median match the plot?

In [ ]:
Subgroup = gapminder[(gapminder["year"] == 1952) &
                     (gapminder["continent"] == "Africa")]

Subgroup["lifeExp"].median()